# GlucoTracker — Notebook consolidado: dataset, features y modelo de riesgo de hipoglucemia

Este notebook junta en un solo flujo lo que antes estaba en 3 notebooks separados, para que se pueda correr de principio a fin sin perder el hilo de las decisiones tomadas en cada paso:

- **Parte A — EDA y construcción del dataset:** de `ShanghaiT2DM_Summary.xlsx` a `dataset_inputs_hipoglucemia.csv` (nivel de lectura: `glucosa` + `momento` + perfil clínico).
- **Parte B — Preparación de features:** codificación de categóricas, manejo de faltantes, split train/test agrupado por paciente real (sin fuga de datos).
- **Parte C y D — Modelos:** Regresión Logística (baseline, justificado) y Random Forest (contraste), cada uno como un **Pipeline de `imblearn`** que integra generación sintética (SMOTE-NC), preprocesamiento y el clasificador en un solo objeto.
- **Parte E — Comparación y conclusiones.**

**Decisiones de alcance ya tomadas y que este notebook respeta:**
- Solo hipoglucemia (no hiperglucemia — se descartó por falta de datasets robustos).
- Una sola toma de glucosa por predicción, sin CGM ni ventanas de tiempo (así es como la app registra datos).
- El motor de reglas ADA (if/else) sigue existiendo aparte, para dar la clasificación instantánea Normal/Hipo/Hiper de cualquier lectura — este notebook entrena el componente de **ML** que da la probabilidad de riesgo, no reemplaza esa regla.
- **HbA1c (%)**: la app hoy no lo captura como parte del flujo normal de registro de glucosa, pero SÍ tiene valor clínico para el riesgo de hipoglucemia. Se decidió mantenerlo como **campo opcional del perfil** — se le pregunta al usuario una sola vez al configurar su perfil ("¿cuál fue tu último HbA1c, si lo conoces?"), igual que edad o comorbilidades. Quien no lo sepa simplemente no bloquea el uso de la app: el modelo ya maneja su ausencia con una bandera `hba1c_faltante` + imputación por mediana (Parte B), así que no es una variable obligatoria de entrada, es una que suma señal cuando está disponible.

---
# PARTE A — EDA y construcción del dataset

## Celda A1 — Montar Drive y cargar el summary

Por qué: todo el resto del notebook depende de tener el Excel del summary accesible en Drive. Se separa en su propia celda para no tener que re-montar Drive cada vez que se reinicia el entorno de ejecución.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = "/content/drive/MyDrive/GlucoTracker/dataset_semilla"
SUMMARY_PATH = f"{BASE_DIR}/ShanghaiT2DM_Summary.xlsx"

df = pd.read_excel(SUMMARY_PATH, sheet_name=0)
df.columns = [str(c).strip() for c in df.columns]

print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head()


## Celda A2 — Limpieza básica y definición del target

Por qué: el dataset usa el texto `"/"` como marcador de dato faltante en vez de dejar la celda vacía — si no se convierte a `NaN`, pandas trata esas columnas como texto y ningún cálculo numérico funciona. También se define aquí el target: se usa la columna real `Hypoglycemia (yes/no)` del dataset tal cual, calculada por los autores a partir de su propio monitoreo CGM — **no se computa nada nuestro**, así que no hay riesgo de que el target termine siendo una función disfrazada de las mismas variables que se usan como input.

In [ ]:
# 1. "/" -> NaN en todo el dataframe
df = df.replace(to_replace=r'^\s*/\s*$', value=np.nan, regex=True)

# 2. Forzar a numerico las columnas que deberian serlo (llegan como texto por el paso anterior)
columnas_numericas = [
    "Age (years)", "Height (m)", "Weight (kg)", "BMI (kg/m2)",
    "Smoking History (pack year)", "Duration of diabetes (years)",
    "Fasting Plasma Glucose (mg/dl)", "2-hour Postprandial Plasma Glucose (mg/dl)",
    "Fasting C-peptide (nmol/L)", "2-hour Postprandial C-peptide (nmol/L)",
    "Fasting Insulin (pmol/L)", "2-hour Postprandial insulin (pmol/L)",
    "HbA1c (%)", "Glycated Albumin (%)",
    "Total Cholesterol (mmol/L)", "Triglyceride (mmol/L)",
    "High-Density Lipoprotein Cholesterol (mmol/L)", "Low-Density Lipoprotein Cholesterol (mmol/L)",
    "Creatinine (umol/L)", "Estimated Glomerular Filtration Rate  (ml/min/1.73m2) ",
    "Uric Acid (mmol/L)", "Blood Urea Nitrogen (mmol/L)",
]
for col in columnas_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 3. Target = columna real del dataset, sin computo nuestro.
#    OJO: esto describe al PACIENTE ("tuvo al menos un episodio real durante su
#    monitoreo"), no a esta fila en particular -- se vuelve importante en la Celda A9.
df["riesgo_hipoglucemia_paciente"] = df["Hypoglycemia (yes/no)"].str.strip().str.lower()

print("Distribucion de riesgo_hipoglucemia_paciente:")
print(df["riesgo_hipoglucemia_paciente"].value_counts())
print(f"\nProporcion de casos positivos: {(df['riesgo_hipoglucemia_paciente']=='yes').mean()*100:.1f}%")


## Celda A3 — Completitud de datos por columna

Por qué: antes de decidir qué variables entran al modelo, hay que saber cuánto falta en cada una. Una columna con 40-50% de datos faltantes no puede ser un input obligatorio de la app — como mucho, un input opcional (como se decidió para HbA1c).

In [ ]:
faltantes = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
faltantes = faltantes[faltantes > 0]

plt.figure(figsize=(8, max(4, len(faltantes)*0.3)))
plt.barh(faltantes.index[::-1], faltantes.values[::-1])
plt.xlabel("% de valores faltantes")
plt.title("Completitud por columna")
plt.tight_layout()
plt.show()

print(faltantes)


## Celda A4 — Detectar columnas sin poder predictivo (varianza cero)

Por qué: si todos (o casi todos) los pacientes tienen el mismo valor en una columna, esa columna no puede ayudar a distinguir quién tiene riesgo y quién no — es ruido, y encima ocupa un campo que la app tendría que pedirle al usuario sin ninguna ganancia a cambio.

In [ ]:
columnas_categoricas_revisar = [
    "Type of Diabetes", "Acute Diabetic Complications",
    "Alcohol Drinking History (drinker/non-drinker)",
]
for col in columnas_categoricas_revisar:
    if col in df.columns:
        print(f"--- {col} ---")
        print(df[col].value_counts(dropna=False))
        print()


## Celda A5 — Simplificar comorbilidades y complicaciones a banderas binarias

Por qué: `Comorbidities` llega como texto libre con combinaciones casi únicas por paciente ("Hypertension, Hyperlipidemia", "Hypertension, CAD", etc.) — en ese formato no sirve como input de un modelo ni como algo que la app le pueda preguntar al usuario con una casilla simple. Se extraen banderas binarias (sí/no) de las comorbilidades que la literatura clínica asocia a riesgo de hipoglucemia (hipertensión, enfermedad renal, neuropatía — ver justificación en la propuesta), que es además el formato que la app SÍ puede pedir fácilmente en un formulario de perfil.

In [ ]:
def limpiar_texto_lista(texto):
    if pd.isna(texto):
        return texto
    partes = [p.strip() for p in str(texto).split(",")]
    partes_unicas = list(dict.fromkeys(p for p in partes if p))
    return ", ".join(partes_unicas)

for col in ["Comorbidities", "Diabetic Macrovascular  Complications", "Diabetic Microvascular Complications"]:
    if col in df.columns:
        df[col] = df[col].apply(limpiar_texto_lista)

def tiene_palabra(texto, palabras):
    if pd.isna(texto):
        return False
    t = str(texto).lower()
    return any(p in t for p in palabras)

df["com_hipertension"] = df["Comorbidities"].apply(lambda t: tiene_palabra(t, ["hypertension"]))
df["com_dislipidemia"] = df["Comorbidities"].apply(lambda t: tiene_palabra(t, ["hyperlipidemia", "dyslipidemia"]))
df["com_renal"] = df["Comorbidities"].apply(lambda t: tiene_palabra(t, ["renal", "kidney", "nephro"]))

if "Diabetic Microvascular Complications" in df.columns:
    df["com_neuropatia"] = df["Diabetic Microvascular Complications"].apply(lambda t: tiene_palabra(t, ["neuropathy"]))
    df["com_nefropatia"] = df["Diabetic Microvascular Complications"].apply(lambda t: tiene_palabra(t, ["nephropathy"]))
if "Diabetic Macrovascular  Complications" in df.columns:
    df["com_cardiovascular"] = df["Diabetic Macrovascular  Complications"].apply(
        lambda t: tiene_palabra(t, ["coronary", "cerebrovascular", "arterial"])
    )

banderas = [c for c in df.columns if c.startswith("com_")]
print("Prevalencia de cada bandera clinica:")
print(df[banderas].mean().round(3) * 100)


## Celda A6 — Simplificar la medicación a categorías de riesgo

Por qué: igual que con comorbilidades, `Hypoglycemic Agents` trae nombres de fármacos específicos (texto libre) que ni un modelo simple ni un formulario de app pueden usar directo. Se reduce a 4 categorías **ordenadas por mecanismo clínico de riesgo**: insulina y sulfonilureas causan hipoglucemia directamente (estimulan o reemplazan la producción de insulina sin importar cuánta glucosa haya en sangre); metformina y otros orales, prácticamente no. Ese orden se reutiliza más adelante al codificar la variable como número (Celda B3).

In [ ]:
def simplificar_agente(texto):
    if pd.isna(texto) or str(texto).strip().lower() == "none":
        return "ninguno"
    t = str(texto).lower()
    palabras_insulina = ["insulin", "novolin", "degludec", "aspart", "glargine", "csii", "lispro", "detemir"]
    palabras_sulfonilurea = ["glimepiride", "glipizide", "gliclazide", "glyburide", "glibenclamide"]
    if any(p in t for p in palabras_insulina):
        return "insulina"
    if any(p in t for p in palabras_sulfonilurea):
        return "sulfonilurea"
    return "metformina_u_otros"

df["categoria_medicacion"] = df["Hypoglycemic Agents"].apply(simplificar_agente)
print(df["categoria_medicacion"].value_counts())


## Celda A7 — EDA univariado: variables numéricas candidatas

Por qué: antes de mirar la relación con el target, conviene ver la forma de cada distribución (¿hay outliers?, ¿está sesgada?) — esto ayuda a decidir después si conviene escalar o transformar antes de meterlas a un modelo lineal.

In [ ]:
numericas_candidatas = ["Age (years)", "BMI (kg/m2)", "Duration of diabetes (years)"]

fig, axes = plt.subplots(1, len(numericas_candidatas), figsize=(14, 4))
for ax, col in zip(axes, numericas_candidatas):
    ax.hist(df[col].dropna(), bins=15)
    ax.set_title(col)
plt.tight_layout()
plt.show()

df[numericas_candidatas].describe()


## Celda A8 — EDA bivariado: cada candidata vs. el target

Por qué: esta es la evidencia real para decidir qué queda como input — no basta con que una variable "suene" clínicamente relevante, hay que ver si en ESTE dataset efectivamente separa a quienes tuvieron hipoglucemia de quienes no. Con n=109 y solo ~17 positivos, ninguna diferencia individual va a ser aplastante — el punto es identificar señal razonable, no encontrar un predictor perfecto por sí solo.

In [ ]:
print("=== Variables numericas: promedio por grupo ===")
print(df.groupby("riesgo_hipoglucemia_paciente")[numericas_candidatas].mean().round(2))

print("\n=== Genero vs target ===")
print(pd.crosstab(df["Gender (Female=1, Male=2)"], df["riesgo_hipoglucemia_paciente"]))

print("\n=== Medicacion vs target ===")
print(pd.crosstab(df["categoria_medicacion"], df["riesgo_hipoglucemia_paciente"]))

print("\n=== Comorbilidades (banderas) vs target ===")
for b in banderas:
    print(f"--- {b} ---")
    print(pd.crosstab(df[b], df["riesgo_hipoglucemia_paciente"]))
    print()


In [ ]:
fig, axes = plt.subplots(1, len(numericas_candidatas), figsize=(14, 4))
for ax, col in zip(axes, numericas_candidatas):
    df.boxplot(column=col, by="riesgo_hipoglucemia_paciente", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
plt.suptitle("")
plt.tight_layout()
plt.show()


## Celda A9 — Expandir a nivel de lectura (agregar la medida de glucosa)

Por qué: hasta aquí el dataset tenía una fila por paciente y le faltaba la variable más importante para GlucoTracker — la lectura de glucosa en sí. El summary trae dos mediciones reales por paciente, `Fasting Plasma Glucose` (en ayunas) y `2-hour Postprandial Plasma Glucose` (postprandial), que son exactamente el tipo de dato que la app registra (glucosa + momento). Se expande el dataset para que cada paciente aporte hasta 2 filas — una por cada lectura real que tenga — ambas con el mismo perfil clínico y el mismo `target`, porque siguen siendo la misma persona con el mismo riesgo de fondo, solo observado en dos momentos distintos del día.

**Por qué esto importa para la tesis:** un mismo paciente puede tener una lectura en ayunas baja y una postprandial alta, y aun así comparte el mismo `target`. Eso demuestra que el riesgo real NO es una función directa del valor de glucosa de esa toma — dos lecturas muy distintas de la misma persona no se explican solo por su número, así que el modelo está obligado a apoyarse en el perfil clínico para resolver la tarea. Es evidencia concreta de que esto no es un umbral disfrazado de ML.

In [ ]:
filas_lectura = []

for _, row in df.iterrows():
    perfil_base = row.to_dict()

    fpg = row.get("Fasting Plasma Glucose (mg/dl)")
    if pd.notna(fpg):
        fila = dict(perfil_base)
        fila["glucosa"] = fpg
        fila["momento"] = "ayunas"
        filas_lectura.append(fila)

    ppg = row.get("2-hour Postprandial Plasma Glucose (mg/dl)")
    if pd.notna(ppg):
        fila = dict(perfil_base)
        fila["glucosa"] = ppg
        fila["momento"] = "postprandial"
        filas_lectura.append(fila)

df_lecturas = pd.DataFrame(filas_lectura)

print(f"Pacientes originales: {df.shape[0]}")
print(f"Filas a nivel de lectura: {df_lecturas.shape[0]}")
print(f"Distribucion de 'momento': {df_lecturas['momento'].value_counts().to_dict()}")
print()
print("Verificacion: mismo paciente, dos lecturas de glucosa distintas, misma etiqueta.")
ejemplo_id = df_lecturas["Patient Number"].iloc[0]
print(df_lecturas[df_lecturas["Patient Number"] == ejemplo_id][["Patient Number", "glucosa", "momento", "riesgo_hipoglucemia_paciente"]])


## Celda A10 — Selección final de columnas y guardado

Por qué: se cierra la Parte A quedándose solo con lo que, según la completitud (A3), la varianza (A4) y la relación con el target (A8), tiene sentido pedirle a un usuario de la app o calcular a partir de lo que ya registra.

- **Obligatorias** (bajo costo, la app las pide directo): `glucosa`, `momento`, género, edad, BMI, duración de diabetes, categoría de medicación, banderas de comorbilidad.
- **Opcional** (requiere un dato de laboratorio que no todos tienen a la mano): `HbA1c (%)` — se pide una sola vez en el perfil, "si lo conoces"; su ausencia se maneja en la Parte B con una bandera + imputación, no bloquea nada.
- **Se descartan:** `Patient Number` (identificador, no predictor), `Type of Diabetes` / `Acute Diabetic Complications` (varianza cero, confirmado en A4), los campos de texto libre ya reemplazados por sus versiones simplificadas, y labs muy incompletos o poco realistas de pedirle a un usuario (C-péptido, insulina de laboratorio, eGFR si su faltante es alto — revisar el % exacto en A3 antes de decidir sobre eGFR).

In [ ]:
columnas_finales = [
    "Patient Number",
    "glucosa", "momento",
    "Gender (Female=1, Male=2)",
    "Age (years)",
    "BMI (kg/m2)",
    "Duration of diabetes (years)",
    "categoria_medicacion",
    "com_hipertension", "com_dislipidemia", "com_renal", "com_neuropatia", "com_nefropatia", "com_cardiovascular",
    "HbA1c (%)",
    "Estimated Glomerular Filtration Rate  (ml/min/1.73m2) ",
    "riesgo_hipoglucemia_paciente",
]
columnas_finales = [c for c in columnas_finales if c in df_lecturas.columns]

df_final = df_lecturas[columnas_finales].copy()
df_final["id_paciente_real"] = df_final["Patient Number"].str.split("_").str[0]

import os
os.makedirs(f"{BASE_DIR}/outputs", exist_ok=True)
OUTPUT_PATH = f"{BASE_DIR}/outputs/dataset_inputs_hipoglucemia.csv"
df_final.to_csv(OUTPUT_PATH, index=False)

print(f"Guardado: {OUTPUT_PATH}")
print(f"Filas: {df_final.shape[0]} | Columnas: {df_final.shape[1]}")
print(f"Pacientes reales unicos: {df_final['id_paciente_real'].nunique()}")
df_final.head()


---
# PARTE B — Preparación de features para entrenamiento

## Celda B1 — Instalar dependencias y recargar el dataset

Por qué: se recarga desde el CSV guardado (en vez de seguir usando `df_final` en memoria) para que esta parte se pueda correr de forma independiente si se reinicia el entorno. `imbalanced-learn` no viene preinstalado en Colab por defecto.

In [ ]:
!pip install -q imbalanced-learn scikit-learn pandas

df = pd.read_csv(f"{BASE_DIR}/outputs/dataset_inputs_hipoglucemia.csv")
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head()


## Celda B2 — Codificar target y variables binarias

Por qué: los modelos de scikit-learn necesitan números, no texto (`"yes"/"no"`) ni booleanos ambiguos. Se codifican aquí las variables que ya son naturalmente binarias — no necesitan ninguna técnica especial, solo un mapeo directo a 0/1.

In [ ]:
df["y"] = (df["riesgo_hipoglucemia_paciente"].str.strip().str.lower() == "yes").astype(int)

if "Gender (Female=1, Male=2)" in df.columns:
    # Gender ya viene 1/2 en el dataset original; se recodifica a 0/1 con un
    # nombre explicito para que nadie confunda "2" con "mas riesgo" o algo asi.
    df["genero_masculino"] = (df["Gender (Female=1, Male=2)"] == 2).astype(int)

for col in [c for c in df.columns if c.startswith("com_")]:
    df[col] = df[col].astype(int)

print("Target codificado:")
print(df["y"].value_counts())
print(f"Proporcion positiva: {df['y'].mean()*100:.1f}%")


## Celda B3 — Codificar categóricas de más de 2 valores

Por qué: `momento` y `categoria_medicacion` tienen más de 2 categorías cada una. Se codifican primero como enteros simples (no one-hot todavía) porque el siguiente paso, SMOTE-NC (Parte C), necesita saber *qué columnas son categóricas* mediante sus índices — eso solo funciona de forma limpia si cada categórica es una sola columna numérica, no un grupo de columnas one-hot ya expandidas.

In [ ]:
mapeo_momento = {"ayunas": 0, "postprandial": 1}
df["momento_cod"] = df["momento"].map(mapeo_momento)

# Mismo orden de riesgo clinico definido en la Celda A6
mapeo_medicacion = {"ninguno": 0, "metformina_u_otros": 1, "sulfonilurea": 2, "insulina": 3}
df["medicacion_cod"] = df["categoria_medicacion"].map(mapeo_medicacion)

print("Mapeo momento:", mapeo_momento)
print("Mapeo medicacion:", mapeo_medicacion)
print()
print(df[["momento", "momento_cod", "categoria_medicacion", "medicacion_cod"]].drop_duplicates())


## Celda B4 — Manejo de HbA1c como campo opcional

Por qué: esta celda es la implementación concreta de la decisión tomada en la introducción. En vez de excluir a los pacientes sin HbA1c (perderíamos filas) o dejar el hueco tal cual (rompe cualquier modelo), se crea una bandera `hba1c_faltante` que le dice al modelo "este dato no se conocía" — y se imputa el valor con la mediana, un valor neutral que no inventa una tendencia. Así el modelo puede aprovechar el HbA1c cuando existe, y no penaliza ni bloquea al usuario que no lo tiene.

In [ ]:
columnas_opcionales_con_nan = [c for c in ["HbA1c (%)", "Estimated Glomerular Filtration Rate  (ml/min/1.73m2) "] if c in df.columns]

for col in columnas_opcionales_con_nan:
    nombre_bandera = col.split(" (")[0].strip().lower().replace(" ", "_") + "_faltante"
    df[nombre_bandera] = df[col].isna().astype(int)
    mediana = df[col].median()
    df[col] = df[col].fillna(mediana)
    print(f"{col}: {df[nombre_bandera].sum()} imputados con mediana={mediana:.2f}")


## Celda B5 — Armar la matriz de features (X, y, grupos)

Por qué: se consolidan en un solo lugar las columnas que van a entrar al modelo. `grupos` (el paciente real, no la fila) se guarda aparte porque el split del siguiente paso necesita agrupar por paciente, no por lectura.

In [ ]:
columnas_numericas_modelo = ["glucosa", "Age (years)", "BMI (kg/m2)", "Duration of diabetes (years)"] + columnas_opcionales_con_nan
columnas_categoricas_modelo = ["momento_cod", "medicacion_cod", "genero_masculino"] + [c for c in df.columns if c.startswith("com_")]
columnas_banderas_faltante = [c for c in df.columns if c.endswith("_faltante")]

columnas_features = columnas_numericas_modelo + columnas_categoricas_modelo + columnas_banderas_faltante
columnas_features = [c for c in columnas_features if c in df.columns]

X = df[columnas_features].copy()
y = df["y"].copy()
grupos = df["id_paciente_real"].copy()

print(f"Features: {len(columnas_features)}")
print(columnas_features)
print(f"\nFilas: {X.shape[0]} | Positivos: {y.sum()} ({y.mean()*100:.1f}%)")


## Celda B6 — Split train/test agrupado por paciente real

Por qué: si el split fuera aleatorio por fila, las dos lecturas (ayunas/postprandial) del mismo paciente podrían quedar una en train y otra en test — el modelo "vería" parcialmente al paciente de prueba antes de ser evaluado con él, inflando artificialmente las métricas. `GroupShuffleSplit` agrupa por `id_paciente_real` para que un paciente completo quede de un solo lado. El test se guarda tal cual, 100% real — la Parte C nunca le va a agregar nada sintético.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=grupos))

X_train, X_test = X.iloc[train_idx].reset_index(drop=True), X.iloc[test_idx].reset_index(drop=True)
y_train, y_test = y.iloc[train_idx].reset_index(drop=True), y.iloc[test_idx].reset_index(drop=True)

print(f"Train: {X_train.shape[0]} filas | Positivos: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Test:  {X_test.shape[0]} filas | Positivos: {y_test.sum()} ({y_test.mean()*100:.1f}%)")

# Verificacion: ningun paciente real debe estar en ambos conjuntos
pacientes_train = set(grupos.iloc[train_idx])
pacientes_test = set(grupos.iloc[test_idx])
interseccion = pacientes_train & pacientes_test
print(f"\nPacientes en ambos conjuntos (debe ser 0): {len(interseccion)}")
assert len(interseccion) == 0, "Fuga de datos: hay pacientes repetidos entre train y test"


---
# PARTE C — Modelo baseline: Regresión Logística (como Pipeline)

## Por qué Regresión Logística como baseline

1. **Tamaño de muestra pequeño.** El dataset real tiene 109 pacientes (17 positivos antes de generar sintéticos). La regla práctica de Peduzzi et al. (1996) sobre "eventos por variable" advierte que modelos complejos con pocos eventos reales tienden a sobreajustar — un modelo lineal con pocos parámetros es más robusto en este régimen.
2. **Interpretabilidad clínica.** Cada coeficiente se traduce a un **odds ratio** ("por cada unidad que sube X, el riesgo se multiplica por Y") — el estándar en modelos de predicción clínica (Steyerberg, *Clinical Prediction Models*, 2019).
3. **Da `predict_proba` de forma nativa.** Es justo lo que GlucoTracker necesita mostrar ("0.7 de probabilidad de hipoglucemia"), sin forzar un modelo pensado para otra cosa.
4. **Es el estándar de comparación en la literatura** de predicción de hipoglucemia con datos tabulares — compararse contra ella es defendible académicamente, no arbitrario.

## Por qué usar un `Pipeline` de `imblearn` (y no hacerlo a mano paso por paso)

En la versión anterior, el orden "codificar → separar train/test → SMOTE-NC solo en train → one-hot" se hacía manualmente, celda por celda — funciona, pero deja espacio a errores si alguien corre las celdas fuera de orden o reutiliza el código en otro lado (por ejemplo, aplicando SMOTE al test por accidente).

Un `imblearn.pipeline.Pipeline` (distinto del `Pipeline` normal de scikit-learn, que no soporta pasos de resampling) resuelve esto por diseño: el paso de `SMOTENC` **solo se ejecuta dentro de `.fit()`**, nunca dentro de `.predict()` o `.transform()`. Eso significa que el mismo objeto `pipeline_lr` se puede entrenar con `fit(X_train, y_train)` y evaluar con `predict(X_test)` sin poder — ni por error — generar sintéticos sobre el test. Es la misma garantía de antes, pero aplicada automáticamente por la librería en vez de depender de que el orden de las celdas se respete.

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Indices de las columnas categoricas dentro de X_train, tal como las necesita SMOTENC
# (tiene que ir ANTES del one-hot: SMOTENC trabaja sobre las categorias ya como
# numeros simples, y genera sintéticos respetando que esas columnas son categoricas,
# no continuas -- si no se le dice esto, "inventaria" valores como momento_cod=0.5).
indices_categoricas = [X_train.columns.get_loc(c) for c in columnas_categoricas_modelo + columnas_banderas_faltante if c in X_train.columns]

# Con tan pocos positivos reales, el split puede dejar muy pocos en train.
# k_neighbors no puede ser mayor o igual a los positivos disponibles.
positivos_train = int(y_train.sum())
if positivos_train < 2:
    raise ValueError(
        f"Muy pocos positivos en train ({positivos_train}) para aplicar SMOTE-NC. "
        "Prueba otro random_state en la Celda B6."
    )
k_neighbors = max(1, min(5, positivos_train - 1))
print(f"Positivos en train: {positivos_train} | k_neighbors: {k_neighbors} | indices categoricos: {indices_categoricas}")

# Preprocesamiento: escalar las numericas (la regresion logistica es sensible a
# la escala) y one-hot a las categoricas de mas de 2 valores. Las binarias
# (genero, com_*, banderas de faltante) ya son 0/1, pasan tal cual con "remainder".
preprocesador = ColumnTransformer(
    transformers=[
        ("escalado", StandardScaler(), columnas_numericas_modelo),
        ("onehot", OneHotEncoder(handle_unknown="ignore"), ["momento_cod", "medicacion_cod"]),
    ],
    remainder="passthrough",
)

pipeline_lr = ImbPipeline(steps=[
    ("smote", SMOTENC(categorical_features=indices_categoricas, random_state=42, k_neighbors=k_neighbors)),
    ("preprocesamiento", preprocesador),
    ("modelo", LogisticRegression(max_iter=1000, random_state=42)),
])

pipeline_lr.fit(X_train, y_train)
print("\nPipeline de Regresion Logistica entrenado.")


## Celda C2 — Evaluar sobre `test_real` (pacientes 100% reales)

Por qué: se llama `pipeline_lr.predict(X_test)` directo sobre los datos originales de test — sin escalar, sin one-hot manual, sin preocuparse de aplicar SMOTE por accidente. El pipeline ya sabe que en modo predicción se salta el paso de `SMOTENC` y solo aplica el mismo preprocesamiento que aprendió del train.

Se usa Recall y F1 de la clase positiva (no accuracy) porque el dataset es desbalanceado (~9% positivos) — un modelo que siempre dijera "no" tendría accuracy alto y sería inútil para el propósito real: detectar a quién sí está en riesgo.

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, precision_recall_fscore_support
)

y_pred_lr = pipeline_lr.predict(X_test)
y_proba_lr = pipeline_lr.predict_proba(X_test)[:, 1]

print("=== Regresion Logistica (Pipeline) — resultados en test_real ===\n")
print(classification_report(y_test, y_pred_lr, target_names=["no_hipo", "hipo"], zero_division=0))

auc_lr = roc_auc_score(y_test, y_proba_lr)
print(f"AUC-ROC: {auc_lr:.3f}")

print("\nMatriz de confusion (filas=real, columnas=prediccion):")
print(pd.DataFrame(
    confusion_matrix(y_test, y_pred_lr),
    index=["real_no", "real_hipo"], columns=["pred_no", "pred_hipo"]
))


## Celda C3 — Interpretar el modelo: odds ratios

Por qué: esto es lo que la regresión logística ofrece y un modelo de caja negra no da tan directo. Como ahora el preprocesamiento vive dentro del pipeline (`ColumnTransformer`), los nombres de las columnas cambian (ej. `onehot__momento_cod_1`) — se recuperan con `get_feature_names_out()` para poder leer la tabla sin perderse.

In [ ]:
nombres_features = pipeline_lr.named_steps["preprocesamiento"].get_feature_names_out()
coeficientes = pipeline_lr.named_steps["modelo"].coef_[0]

odds_ratios = pd.DataFrame({
    "feature": nombres_features,
    "coeficiente": coeficientes,
    "odds_ratio": np.exp(coeficientes),
}).sort_values("odds_ratio", ascending=False)

print("Odds ratios (>1 = aumenta el riesgo, <1 = lo reduce):")
odds_ratios


---
# PARTE D — Segundo modelo: Random Forest (mismo esquema de Pipeline)

## Celda D1 — Entrenar Random Forest

Por qué: se entrena con la misma estructura de pipeline (SMOTE-NC + preprocesamiento + modelo) para que la comparación con el baseline sea justa — mismos datos, mismo split, mismo manejo de sintéticos. Random Forest no necesita el escalado de `StandardScaler` (es invariante a transformaciones monótonas de las variables), pero no hace daño reutilizar el mismo `ColumnTransformer` por consistencia y simplicidad del código.

`max_depth` se limita a propósito: con tan pocos pacientes reales, un Random Forest sin límite de profundidad memoriza el train en vez de generalizar. `class_weight="balanced"` es una segunda red de seguridad además del SMOTE, para que el árbol no ignore la clase minoritaria.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

pipeline_rf = ImbPipeline(steps=[
    ("smote", SMOTENC(categorical_features=indices_categoricas, random_state=42, k_neighbors=k_neighbors)),
    ("preprocesamiento", preprocesador),
    ("modelo", RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=3,
        random_state=42,
        class_weight="balanced",
    )),
])

pipeline_rf.fit(X_train, y_train)

y_pred_rf = pipeline_rf.predict(X_test)
y_proba_rf = pipeline_rf.predict_proba(X_test)[:, 1]

print("=== Random Forest (Pipeline) — resultados en test_real ===\n")
print(classification_report(y_test, y_pred_rf, target_names=["no_hipo", "hipo"], zero_division=0))

auc_rf = roc_auc_score(y_test, y_proba_rf)
print(f"AUC-ROC: {auc_rf:.3f}")

print("\nMatriz de confusion (filas=real, columnas=prediccion):")
print(pd.DataFrame(
    confusion_matrix(y_test, y_pred_rf),
    index=["real_no", "real_hipo"], columns=["pred_no", "pred_hipo"]
))


## Celda D2 — Importancia de variables en Random Forest

Por qué: sirve de contraste con los odds ratios de la Parte C — no dice la dirección del efecto (si sube o baja el riesgo), pero sí cuánto "pesa" cada variable para que el modelo separe las clases.

In [ ]:
nombres_features_rf = pipeline_rf.named_steps["preprocesamiento"].get_feature_names_out()
importancias = pd.DataFrame({
    "feature": nombres_features_rf,
    "importancia": pipeline_rf.named_steps["modelo"].feature_importances_,
}).sort_values("importancia", ascending=False)

importancias


---
# PARTE E — Comparación final y conclusiones

## Celda E1 — Tabla comparativa

Por qué: reunir las métricas de ambos pipelines en una sola tabla es lo que realmente se reporta en la tesis — no basta con mostrar cada modelo por separado, hay que poder decir con cuál queda GlucoTracker y por qué.

In [ ]:
precision_lr, recall_lr, f1_lr, _ = precision_recall_fscore_support(y_test, y_pred_lr, average="binary", zero_division=0)
precision_rf, recall_rf, f1_rf, _ = precision_recall_fscore_support(y_test, y_pred_rf, average="binary", zero_division=0)

comparacion = pd.DataFrame({
    "modelo": ["Regresion Logistica (baseline)", "Random Forest"],
    "precision_hipo": [precision_lr, precision_rf],
    "recall_hipo": [recall_lr, recall_rf],
    "f1_hipo": [f1_lr, f1_rf],
    "auc_roc": [auc_lr, auc_rf],
}).round(3)

comparacion


## Conclusión

- El pipeline de **Regresión Logística** es el baseline justificado por tamaño de muestra, interpretabilidad (odds ratios) y por dar `predict_proba` de forma nativa — que es exactamente lo que GlucoTracker necesita mostrar al usuario.
- El pipeline de **Random Forest** se prueba como contraste: si mejora notablemente Recall/F1 sobre el baseline, es evidencia de interacciones no lineales entre variables (ej. medicación + comorbilidades) que el modelo lineal no captura. Si no mejora, es evidencia de que con esta cantidad de datos reales el modelo simple ya es suficiente — y preferible, por interpretabilidad.
- Usar `imblearn.pipeline.Pipeline` en ambos casos no es solo prolijidad de código: es lo que garantiza, por construcción, que SMOTE-NC nunca toca el conjunto de test ni se aplicaría por error en producción si este mismo pipeline se reutiliza para predecir sobre un paciente nuevo.
- **Con tan pocos pacientes reales de test**, estas métricas tienen varianza alta — no deberían leerse como una medición definitiva de qué modelo es "mejor". El siguiente paso natural es validación cruzada agrupada por paciente (`GroupKFold`) para tener una estimación más estable antes de elegir el modelo final para GlucoTracker.
- **Limitación de alcance documentada:** el modelo cubre los momentos `ayunas` y `postprandial` únicamente — `antes de dormir` no tiene un valor real equivalente en este dataset, así que para esa franja el sistema sigue apoyándose solo en el motor de reglas ADA hasta conseguir datos que la cubran.